# Kaggle GPU × Ollama × ngrok

Kaggleの無料GPU（T4 × 2枚）でAIモデルを起動し、ngrokで外部公開するNotebookです。

## 使い方
1. KaggleでこのNotebookをインポート
2. Acceleratorを `GPU T4 x2` に設定
3. Secretsに `NGROK_AUTHTOKEN` を設定
4. セルを上から順に実行

In [ ]:
# セル1: 依存ツールのインストールとGPU確認
!apt-get update && apt-get install -y zstd pciutils
!nvidia-smi

In [ ]:
# セル2: Ollamaのインストールと外部公開設定
import os
import subprocess
import time

# Ollamaのインストール
!curl -fsSL https://ollama.com/install.sh | sh

# サーバー起動設定
OLLAMA_PATH = "/usr/local/bin/ollama"
custom_env = os.environ.copy()
custom_env["OLLAMA_HOST"] = "[::]:11434"
custom_env["OLLAMA_ORIGINS"] = "*"
custom_env["OLLAMA_KEEP_ALIVE"] = "-1"

# 既存プロセスを掃除して起動
subprocess.run(["pkill", "-9", "ollama"], check=False)
with open("ollama.log", "w") as log_file:
    subprocess.Popen([OLLAMA_PATH, "serve"], env=custom_env, stdout=log_file, stderr=log_file)

print("Waiting for Ollama server to start...")
time.sleep(12)
!/usr/local/bin/ollama --version

In [ ]:
# セル3: GGUFモデルの登録とngrok公開
# GGUFモデルのパス（Dataset名に合わせて変更）
GGUF_PATH = "/kaggle/input/qwen3-27b-uncensored-gguf/Qwen3.8-27B-Uncensored-Q4_K_M.gguf"

# Modelfileを作成
modelfile_content = f"""FROM {GGUF_PATH}
TEMPLATE \"\"\"{{ .System }}{{ .Prompt }}\"\"\"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
"""

with open("/tmp/Modelfile", "w") as f:
    f.write(modelfile_content)

# Ollamaモデルとして登録
!/usr/local/bin/ollama create qwen3-27b-uncensored -f /tmp/Modelfile

# ngrokで公開
!pip install -q pyngrok
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
NGROK_TOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")

BASIC_AUTH_USER = "your_username"
BASIC_AUTH_PASS = "your_password"

ngrok.set_auth_token(NGROK_TOKEN)

try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)

    public_url = ngrok.connect(11434, auth=f"{BASIC_AUTH_USER}:{BASIC_AUTH_PASS}")
    print("=== API Server Information ===")
    print(f"Public URL: {public_url}")
    print(f"Auth: {BASIC_AUTH_USER} / {BASIC_AUTH_PASS}")
    print(f"Model: qwen3-27b-uncensored")
    print(f"Test Cmd: curl -u {BASIC_AUTH_USER}:{BASIC_AUTH_PASS} -H \"ngrok-skip-browser-warning: 1\" {public_url}/api/tags")
    print("==============================")
except Exception as e:
    print(f"ngrok Error: {e}")
